# 📊 Visualización Extensiva y Enriquecimiento de Datos
## Fundación Teletón - Dashboard con Plotly Express

**Curso:** CD2001B - Diagnóstico para Líneas de Acción  
**Semana 4:** Visualización de Datos  
**Dataset:** 274 empresas benefactoras de Fundación Teletón  
**Objetivo:** Crear ~50+ visualizaciones con Plotly Express, enriquecer datos, y preparar para Looker Studio/BigQuery/Tableau

---

## 📋 Índice del Notebook

1. ✅ **Setup y Carga** (COMPLETO)
2. ✅ **Diccionario de Datos** (COMPLETO)
3. 🟡 **Enriquecimiento de Datos** (TODO: Iteración 6)
4. 🟡 **Visualizaciones Univariadas** (TODO: Iteración 7)
5. 🟡 **Visualizaciones Bivariadas** (TODO: Iteración 8)
6. 🟡 **Heatmaps y Correlaciones** (TODO: Iteración 9)
7. 🟡 **Gráficos de Distribución Avanzados** (TODO: Iteración 9)
8. 🟡 **Gráficos de Series y Comparaciones** (TODO: Iteración 10)
9. 🟡 **Gráficos de Burbuja y 3D** (TODO: Iteración 10)
10. 🟡 **Gráficos de Tendencia** (TODO: Iteración 10)
11. 🟡 **Gráficos Estadísticos Avanzados** (TODO: Iteración 11)
12. 🟡 **Dashboards Interactivos con Subplots** (TODO: Iteración 11)
13. 🟡 **BONUS: Profilers Automáticos** (TODO: Iteración 12)
14. 🟡 **Exportación para BI Tools** (TODO: Iteración 12)
15. 🟡 **Conclusiones y Próximos Pasos** (TODO: Iteración 12)

---
# ✅ SECCIÓN 1: Setup y Carga de Datos

In [ ]:
# Imports para visualización con Plotly
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import warnings
warnings.filterwarnings('ignore')

print("📦 Librerías de visualización importadas")
print(f"   - pandas v{pd.__version__}")
print(f"   - numpy v{np.__version__}")
print(f"   - plotly v{px.__version__}")

In [ ]:
# Configuración de Plotly
import plotly.io as pio
pio.templates.default = "plotly_white"

# Configuración de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

print("✅ Configuración de Plotly aplicada")
print("   - Template: plotly_white")
print("   - Modo: Interactivo")

In [ ]:
# Cargar datos limpios del Notebook 1
# NOTA: Si aún no has ejecutado el Notebook 1, carga directamente desde Excel

try:
    # Intentar cargar datos procesados del Notebook 1
    df = pd.read_csv('datos_procesados/teleton_enriched.csv')
    print("✅ Datos cargados desde: datos_procesados/teleton_enriched.csv")
except FileNotFoundError:
    # Si no existe, cargar desde Excel original
    print("⚠️ Archivo procesado no encontrado. Cargando desde Excel original...")
    excel_path = '/mnt/c/Users/HG_Co/OneDrive/Documents/Github/diagnostico-lineas-accion/proyecto_reto/teleton.xlsx'
    df = pd.read_excel(excel_path, sheet_name='Base de datos')
    
    # Imputar FI_3 si tiene valores faltantes
    if df['FI_3'].isnull().sum() > 0:
        df['FI_3'].fillna(df['FI_3'].median(), inplace=True)
        df['FI_3'] = df['FI_3'].astype('int64')
    
    print("✅ Datos cargados desde Excel")

print(f"\nDimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")

In [ ]:
# Convertir categóricas a tipo 'category'
categorical_cols = ['Giro', 'Puesto', 'Estado']

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

print("✅ Variables categóricas convertidas a tipo 'category'")

In [ ]:
# Vista rápida de los datos
print("\n📋 PRIMERAS 5 FILAS")
display(df.head())

---
# ✅ SECCIÓN 2: Diccionario de Datos

In [ ]:
# Cargar diccionario de variables desde Excel
excel_path = '/mnt/c/Users/HG_Co/OneDrive/Documents/Github/diagnostico-lineas-accion/proyecto_reto/teleton.xlsx'
dict_variables = pd.read_excel(excel_path, sheet_name='Descripción de variables')

print("="*80)
print("DICCIONARIO DE VARIABLES - FUNDACIÓN TELETÓN")
print("="*80)
display(dict_variables)

## 📊 Resumen de Variables

### Dimensiones SERVQUAL (Escala 1-5)
- **Tangibles (AT):** AT_1, AT_2
- **Fiabilidad (FI):** FI_1, FI_2, FI_3
- **Respuesta (R):** R_1, R_2, R_3
- **Empatía (E):** E_1, E_2, E_3, E_4

### Variables de Desempeño
- **D_1:** Satisfacción (1-10)
- **R_12:** NPS / Recomendación (1-10)
- **C_1:** Calidad percibida (1-5)
- **Info:** Nivel de información (1-10)

### Variables Demográficas
- **Años:** Antigüedad como benefactor
- **Giro:** Sector empresarial (6 categorías)
- **Puesto:** Rol del encuestado (7 categorías)
- **Estado:** Entidad federativa (27 presentes)

**Total:** 20 variables originales × 274 registros

---
# ✅ SECCIÓN 3: Enriquecimiento de Datos

**Objetivo:** Crear variables derivadas para facilitar visualización y análisis avanzado.

Esta sección creará ~10 variables nuevas que permitirán:
- Categorizar variables continuas
- Crear índices y scores
- Agrupar geográficamente
- Facilitar visualizaciones interactivas

## Contenido:
1. Crear dimensiones SERVQUAL (si no existen)
2. Categorizar Satisfacción, Años, Calidad
3. Crear índices (Excelencia, NPS Score, Gap)
4. Clasificar por Regiones geográficas
5. Calcular consistencia de respuestas

In [ ]:
## 3.1 Crear Dimensiones SERVQUAL (si no existen)

print("="*80)
print("VERIFICAR Y CREAR DIMENSIONES SERVQUAL")
print("="*80)

# Verificar si ya existen (del Notebook 1)
dimensiones_servqual = ['Tangibles', 'Fiabilidad', 'Respuesta', 'Empatia', 'SERVQUAL_Total']

if not all(dim in df.columns for dim in dimensiones_servqual):
    print("\n⚠️ Dimensiones SERVQUAL no encontradas. Creando...")
    
    # Crear dimensiones como promedios
    df['Tangibles'] = df[['AT_1', 'AT_2']].mean(axis=1)
    df['Fiabilidad'] = df[['FI_1', 'FI_2', 'FI_3']].mean(axis=1)
    df['Respuesta'] = df[['R_1', 'R_2', 'R_3']].mean(axis=1)
    df['Empatia'] = df[['E_1', 'E_2', 'E_3', 'E_4']].mean(axis=1)
    df['SERVQUAL_Total'] = df[['Tangibles', 'Fiabilidad', 'Respuesta', 'Empatia']].mean(axis=1)
    
    print("✅ Dimensiones SERVQUAL creadas")
else:
    print("\n✅ Dimensiones SERVQUAL ya existen (cargadas desde Notebook 1)")

# Verificar NPS_Categoria
if 'NPS_Categoria' not in df.columns:
    print("\n⚠️ NPS_Categoria no encontrada. Creando...")
    
    def categorizar_nps(valor):
        if valor >= 9:
            return 'Promotor'
        elif valor >= 7:
            return 'Pasivo'
        else:
            return 'Detractor'
    
    df['NPS_Categoria'] = df['R_12'].apply(categorizar_nps)
    print("✅ NPS_Categoria creada")
else:
    print("\n✅ NPS_Categoria ya existe")

print(f"\nColumnas actuales: {df.shape[1]}")

In [ ]:
## 3.10 Resumen del Enriquecimiento

print("="*80)
print("RESUMEN DE ENRIQUECIMIENTO - SECCIÓN 3 COMPLETADA")
print("="*80)

print("\n✅ VARIABLES CREADAS:")

nuevas_variables = [
    'Tangibles', 'Fiabilidad', 'Respuesta', 'Empatia', 'SERVQUAL_Total',
    'NPS_Categoria', 'Categoria_Satisfaccion', 'Grupo_Antiguedad',
    'Nivel_Calidad', 'Indice_Excelencia', 'NPS_Score',
    'Gap_Satisfaccion', 'Consistencia_Respuestas', 'Region'
]

variables_realmente_nuevas = [var for var in nuevas_variables if var in df.columns]

print(f"\nTotal de variables derivadas: {len(variables_realmente_nuevas)}")
for idx, var in enumerate(variables_realmente_nuevas, 1):
    print(f"   {idx}. {var}")

print(f"\n📊 DATASET ENRIQUECIDO:")
print(f"   • Columnas originales: 20")
print(f"   • Columnas derivadas: {len(variables_realmente_nuevas)}")
print(f"   • Total columnas: {df.shape[1]}")
print(f"   • Total filas: {df.shape[0]}")

print(f"\n💾 TAMAÑO EN MEMORIA:")
memoria_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"   {memoria_mb:.2f} MB")

print("\n🎯 VARIABLES CATEGÓRICAS CREADAS:")
categoricas_nuevas = ['Categoria_Satisfaccion', 'Grupo_Antiguedad', 'Nivel_Calidad', 
                       'NPS_Categoria', 'Region']
for var in categoricas_nuevas:
    if var in df.columns:
        print(f"   • {var}: {df[var].nunique()} categorías")

print("\n🎯 ÍNDICES Y SCORES CREADOS:")
indices_nuevos = ['Indice_Excelencia', 'NPS_Score', 'Gap_Satisfaccion', 'Consistencia_Respuestas']
for var in indices_nuevos:
    if var in df.columns:
        print(f"   • {var}: {df[var].min():.2f} - {df[var].max():.2f}")

print("\n" + "="*80)
print("🚀 LISTO PARA ITERACIÓN 7: Visualizaciones Univariadas")
print("="*80)

# Mostrar muestra del dataset enriquecido
print("\n📋 MUESTRA DEL DATASET ENRIQUECIDO (primeras 3 filas, variables clave):")
columnas_mostrar = ['D_1', 'Categoria_Satisfaccion', 'Años', 'Grupo_Antiguedad', 
                     'SERVQUAL_Total', 'Indice_Excelencia', 'NPS_Categoria', 'Region']
display(df[columnas_mostrar].head(3))

In [ ]:
## 3.9 Clasificar por Regiones Geográficas

print("\n" + "="*80)
print("CLASIFICAR ESTADOS POR REGIÓN")
print("="*80)

# Mapeo de Estados a Regiones de México
regiones_mexico = {
    # Norte
    'Baja California': 'Norte',
    'Baja California Sur': 'Norte',
    'Sonora': 'Norte',
    'Chihuahua': 'Norte',
    'Coahuila': 'Norte',
    'Nuevo León': 'Norte',
    'Tamaulipas': 'Norte',
    'Durango': 'Norte',
    'Sinaloa': 'Norte',
    
    # Centro
    'Ciudad de México': 'Centro',
    'México': 'Centro',
    'Hidalgo': 'Centro',
    'Tlaxcala': 'Centro',
    'Puebla': 'Centro',
    'Morelos': 'Centro',
    'Querétaro': 'Centro',
    'Guanajuato': 'Centro',
    'Aguascalientes': 'Centro',
    'San Luis Potosí': 'Centro',
    'Zacatecas': 'Centro',
    
    # Occidente
    'Jalisco': 'Occidente',
    'Colima': 'Occidente',
    'Michoacán': 'Occidente',
    'Nayarit': 'Occidente',
    
    # Sur
    'Guerrero': 'Sur',
    'Oaxaca': 'Sur',
    'Chiapas': 'Sur',
    'Veracruz': 'Sur',
    'Tabasco': 'Sur',
    
    # Península
    'Campeche': 'Península',
    'Yucatán': 'Península',
    'Quintana Roo': 'Península'
}

# Aplicar mapeo
df['Region'] = df['Estado'].map(regiones_mexico)

# Manejar estados no mapeados (si existen)
estados_sin_region = df[df['Region'].isnull()]['Estado'].unique()
if len(estados_sin_region) > 0:
    print(f"\n⚠️ Estados sin región asignada: {list(estados_sin_region)}")
    # Asignar a "Otro" o a la región más cercana manualmente
    df['Region'].fillna('Otro', inplace=True)

# Convertir a categorical
df['Region'] = df['Region'].astype('category')

print("✅ Region creada")
print(f"\nDistribución por Región:")
print(df['Region'].value_counts())

In [ ]:
## 3.8 Calcular Consistencia de Respuestas

print("\n" + "="*80)
print("CALCULAR CONSISTENCIA DE RESPUESTAS")
print("="*80)

# Consistencia = Desviación estándar de todas las respuestas Likert por fila
# Menor std = Mayor consistencia (respuestas homogéneas)
likert_vars = ['AT_1', 'AT_2', 'FI_1', 'FI_2', 'FI_3', 'R_1', 'R_2', 'R_3',
               'E_1', 'E_2', 'E_3', 'E_4', 'C_1']

df['Consistencia_Respuestas'] = df[likert_vars].std(axis=1)

print("✅ Consistencia_Respuestas creada")
print(f"\nEstadísticas (desv. std. de respuestas Likert):")
print(f"   Promedio: {df['Consistencia_Respuestas'].mean():.3f}")
print(f"   Min: {df['Consistencia_Respuestas'].min():.3f} (muy consistente)")
print(f"   Max: {df['Consistencia_Respuestas'].max():.3f} (respuestas variadas)")

print(f"\n💡 Baja consistencia puede indicar respuestas más reflexivas o experiencias mixtas")

In [ ]:
## 3.7 Crear Gap de Satisfacción

print("\n" + "="*80)
print("CREAR GAP DE SATISFACCIÓN")
print("="*80)

# Gap = Máximo posible (10) - Satisfacción actual
# Cuanto mayor el gap, mayor la oportunidad de mejora
df['Gap_Satisfaccion'] = 10 - df['D_1']

print("✅ Gap_Satisfaccion creada")
print(f"\nEstadísticas:")
print(f"   Promedio: {df['Gap_Satisfaccion'].mean():.2f} puntos de mejora")
print(f"   Min: {df['Gap_Satisfaccion'].min():.0f} (satisfacción perfecta)")
print(f"   Max: {df['Gap_Satisfaccion'].max():.0f} (mayor oportunidad)")
print(f"\n💡 Gap promedio de {df['Gap_Satisfaccion'].mean():.2f} puntos indica oportunidad de mejora")

In [ ]:
## 3.6 Crear Score NPS Individual (-1, 0, +1)

print("\n" + "="*80)
print("CREAR SCORE NPS INDIVIDUAL")
print("="*80)

# Convertir categoría a score numérico
def nps_a_score(categoria):
    if categoria == 'Promotor':
        return 1
    elif categoria == 'Pasivo':
        return 0
    else:  # Detractor
        return -1

df['NPS_Score'] = df['NPS_Categoria'].apply(nps_a_score)

print("✅ NPS_Score creada (-1, 0, +1)")
print(f"\nDistribución:")
print(df['NPS_Score'].value_counts().sort_index())

# Calcular NPS agregado
nps_agregado = (df['NPS_Score'].sum() / len(df)) * 100
print(f"\n📊 NPS Agregado del dataset: {nps_agregado:.2f}%")

In [ ]:
## 3.5 Crear Índice de Excelencia en Servicio (0-100)

print("\n" + "="*80)
print("CREAR ÍNDICE DE EXCELENCIA EN SERVICIO")
print("="*80)

# Convertir SERVQUAL_Total (escala 1-5) a índice 0-100
# Fórmula: ((SERVQUAL - 1) / 4) * 100
df['Indice_Excelencia'] = ((df['SERVQUAL_Total'] - 1) / 4) * 100

print("✅ Indice_Excelencia creada (escala 0-100)")
print(f"\nEstadísticas:")
print(f"   Media: {df['Indice_Excelencia'].mean():.2f}")
print(f"   Min: {df['Indice_Excelencia'].min():.2f}")
print(f"   Max: {df['Indice_Excelencia'].max():.2f}")
print(f"   Mediana: {df['Indice_Excelencia'].median():.2f}")

In [ ]:
## 3.4 Categorizar Calidad Percibida (C_1)

print("\n" + "="*80)
print("CATEGORIZAR CALIDAD PERCIBIDA (C_1)")
print("="*80)

# Niveles de Calidad (escala 1-5):
# Deficiente: 1-2, Aceptable: 3, Bueno: 4, Excelente: 5
def categorizar_calidad(valor):
    if valor <= 2:
        return 'Deficiente'
    elif valor == 3:
        return 'Aceptable'
    elif valor == 4:
        return 'Bueno'
    else:  # 5
        return 'Excelente'

df['Nivel_Calidad'] = df['C_1'].apply(categorizar_calidad)

# Convertir a categorical ordenado
df['Nivel_Calidad'] = pd.Categorical(
    df['Nivel_Calidad'],
    categories=['Deficiente', 'Aceptable', 'Bueno', 'Excelente'],
    ordered=True
)

print("✅ Nivel_Calidad creada")
print(f"\nDistribución:")
print(df['Nivel_Calidad'].value_counts().sort_index())

In [ ]:
## 3.3 Categorizar Antigüedad (Años)

print("\n" + "="*80)
print("CATEGORIZAR ANTIGÜEDAD (AÑOS)")
print("="*80)

# Grupos de Antigüedad:
# Nuevo: ≤2, Intermedio: 3-5, Fiel: 6-10, Muy Fiel: >10
def categorizar_antiguedad(años):
    if años <= 2:
        return 'Nuevo'
    elif años <= 5:
        return 'Intermedio'
    elif años <= 10:
        return 'Fiel'
    else:
        return 'Muy Fiel'

df['Grupo_Antiguedad'] = df['Años'].apply(categorizar_antiguedad)

# Convertir a categorical ordenado
df['Grupo_Antiguedad'] = pd.Categorical(
    df['Grupo_Antiguedad'],
    categories=['Nuevo', 'Intermedio', 'Fiel', 'Muy Fiel'],
    ordered=True
)

print("✅ Grupo_Antiguedad creada")
print(f"\nDistribución:")
print(df['Grupo_Antiguedad'].value_counts().sort_index())

In [ ]:
## 3.2 Categorizar Satisfacción (D_1)

print("\n" + "="*80)
print("CATEGORIZAR SATISFACCIÓN (D_1)")
print("="*80)

# Categorías de Satisfacción:
# Baja: 1-6, Media: 7-8, Alta: 9-10
def categorizar_satisfaccion(valor):
    if valor >= 9:
        return 'Alta'
    elif valor >= 7:
        return 'Media'
    else:
        return 'Baja'

df['Categoria_Satisfaccion'] = df['D_1'].apply(categorizar_satisfaccion)

# Convertir a categorical ordenado
df['Categoria_Satisfaccion'] = pd.Categorical(
    df['Categoria_Satisfaccion'],
    categories=['Baja', 'Media', 'Alta'],
    ordered=True
)

print("✅ Categoria_Satisfaccion creada")
print(f"\nDistribución:")
print(df['Categoria_Satisfaccion'].value_counts().sort_index())

---
# ✅ SECCIÓN 4: Visualizaciones Univariadas con Plotly

**Objetivo:** Crear ~16 visualizaciones interactivas de variables individuales usando Plotly Express.

Esta sección analiza la distribución de cada variable por separado, identificando patrones, valores atípicos, y tendencias centrales.

## Contenido:
1. **Histogramas** (4 gráficos): Distribuciones de variables continuas
2. **Box Plots** (2 gráficos): Identificación de outliers y estadísticos
3. **Violin Plots** (2 gráficos): Distribución + densidad
4. **Gráficos de Barras** (3 gráficos): Frecuencias categóricas
5. **Pie/Donut Charts** (2 gráficos): Composición porcentual
6. **Treemaps** (2 gráficos): Jerarquías proporcionales
7. **Sunburst** (1 gráfico): Jerarquía multinivel

**Total:** 16 visualizaciones univariadas interactivas

In [ ]:
## 4.1 HISTOGRAMAS (4 gráficos)

print("="*80)
print("SECCIÓN 4.1: HISTOGRAMAS - DISTRIBUCIONES DE VARIABLES CONTINUAS")
print("="*80)

# Gráfico 1: Distribución de Satisfacción (D_1)
fig1 = px.histogram(
    df, 
    x='D_1',
    nbins=10,
    title='📊 Distribución de Satisfacción General (D_1)',
    labels={'D_1': 'Satisfacción (1-10)', 'count': 'Frecuencia'},
    color_discrete_sequence=['#1f77b4'],
    marginal='box'  # Agregar box plot en el margen
)
fig1.update_layout(
    showlegend=False,
    height=400,
    bargap=0.1
)
fig1.add_vline(x=df['D_1'].mean(), line_dash="dash", line_color="red", 
               annotation_text=f"Media: {df['D_1'].mean():.2f}")
fig1.show()

# Gráfico 2: Distribución de Años (asimétrica)
fig2 = px.histogram(
    df,
    x='Años',
    nbins=20,
    title='📅 Distribución de Antigüedad como Benefactor (Años)',
    labels={'Años': 'Años como benefactor', 'count': 'Frecuencia'},
    color_discrete_sequence=['#2ca02c'],
    marginal='violin'  # Agregar violin plot en el margen
)
fig2.update_layout(
    showlegend=False,
    height=400,
    bargap=0.1
)
fig2.add_vline(x=df['Años'].median(), line_dash="dash", line_color="red",
               annotation_text=f"Mediana: {df['Años'].median():.1f} años")
fig2.show()

# Gráfico 3: Distribución de NPS (R_12)
fig3 = px.histogram(
    df,
    x='R_12',
    nbins=10,
    title='🎯 Distribución de Net Promoter Score (R_12)',
    labels={'R_12': 'NPS (1-10)', 'count': 'Frecuencia'},
    color_discrete_sequence=['#ff7f0e'],
    marginal='rug'  # Agregar rug plot
)
fig3.update_layout(
    showlegend=False,
    height=400,
    bargap=0.1
)
fig3.add_vline(x=9, line_dash="dot", line_color="green",
               annotation_text="Umbral Promotor (≥9)")
fig3.add_vline(x=7, line_dash="dot", line_color="orange",
               annotation_text="Umbral Pasivo (≥7)")
fig3.show()

# Gráfico 4: Distribución de Índice de Excelencia
fig4 = px.histogram(
    df,
    x='Indice_Excelencia',
    nbins=20,
    title='⭐ Distribución del Índice de Excelencia en Servicio (0-100)',
    labels={'Indice_Excelencia': 'Índice de Excelencia (%)', 'count': 'Frecuencia'},
    color_discrete_sequence=['#9467bd'],
    marginal='box'
)
fig4.update_layout(
    showlegend=False,
    height=400,
    bargap=0.1
)
fig4.add_vline(x=df['Indice_Excelencia'].mean(), line_dash="dash", line_color="red",
               annotation_text=f"Media: {df['Indice_Excelencia'].mean():.1f}%")
fig4.show()

print("\n✅ 4 histogramas completados")

---
# ✅ SECCIÓN 5: Visualizaciones Bivariadas

**Objetivo:** Crear ~14 visualizaciones que muestran relaciones entre dos variables.

Esta sección analiza cómo se relacionan pares de variables, identificando correlaciones, diferencias entre grupos, y patrones de asociación.

## Contenido:
1. **Scatter Plots con Regresión** (3 gráficos): Relaciones lineales con trendlines
2. **Scatter Matrix** (1 gráfico): Matriz de dispersión multivariada
3. **Box Plots por Grupos** (4 gráficos): Comparación de distribuciones
4. **Violin Plots por Grupos** (2 gráficos): Densidades comparativas
5. **Barras Agrupadas** (2 gráficos): Comparaciones categóricas
6. **Barras Apiladas** (1 gráfico): Composición proporcional
7. **Strip Plots** (1 gráfico): Distribución individual por categoría

**Total:** 14 visualizaciones bivariadas interactivas

In [ ]:
## 5.1 SCATTER PLOTS CON REGRESIÓN (3 gráficos)

print("="*80)
print("SECCIÓN 5.1: SCATTER PLOTS CON TRENDLINES - RELACIONES LINEALES")
print("="*80)

# Gráfico 1: Empatía vs Satisfacción (con trendline OLS)
fig1 = px.scatter(
    df,
    x='Empatia',
    y='D_1',
    title='💙 Relación: Empatía vs Satisfacción General (con regresión OLS)',
    labels={'Empatia': 'Empatía (1-5)', 'D_1': 'Satisfacción (1-10)'},
    trendline='ols',  # Ordinary Least Squares regression
    color='Giro',
    hover_data=['Estado', 'Años'],
    opacity=0.7
)
fig1.update_layout(height=500)
fig1.update_traces(marker=dict(size=8))
fig1.show()

# Mostrar coeficiente de correlación
corr_empatia_sat = df[['Empatia', 'D_1']].corr().iloc[0, 1]
print(f"\n📊 Correlación Empatía-Satisfacción: {corr_empatia_sat:.3f}")

# Gráfico 2: Años vs NPS (con trendline)
fig2 = px.scatter(
    df,
    x='Años',
    y='R_12',
    title='📅 Relación: Antigüedad vs NPS (con regresión OLS)',
    labels={'Años': 'Años como benefactor', 'R_12': 'NPS (1-10)'},
    trendline='ols',
    color='NPS_Categoria',
    color_discrete_map={'Detractor': '#d62728', 'Pasivo': '#ff7f0e', 'Promotor': '#2ca02c'},
    hover_data=['Giro', 'Estado'],
    opacity=0.7
)
fig2.update_layout(height=500)
fig2.update_traces(marker=dict(size=8))
fig2.show()

corr_años_nps = df[['Años', 'R_12']].corr().iloc[0, 1]
print(f"📊 Correlación Años-NPS: {corr_años_nps:.3f}")

# Gráfico 3: SERVQUAL Total vs Satisfacción (con trendline LOWESS)
fig3 = px.scatter(
    df,
    x='SERVQUAL_Total',
    y='D_1',
    title='⭐ Relación: SERVQUAL Total vs Satisfacción (con regresión LOWESS)',
    labels={'SERVQUAL_Total': 'SERVQUAL Total (1-5)', 'D_1': 'Satisfacción (1-10)'},
    trendline='lowess',  # Locally Weighted Scatterplot Smoothing (no lineal)
    color='Categoria_Satisfaccion',
    color_discrete_map={'Baja': '#d62728', 'Media': '#ff7f0e', 'Alta': '#2ca02c'},
    hover_data=['Giro', 'Años', 'NPS_Categoria'],
    opacity=0.7,
    size='Indice_Excelencia'  # Tamaño por índice de excelencia
)
fig3.update_layout(height=500)
fig3.show()

corr_servqual_sat = df[['SERVQUAL_Total', 'D_1']].corr().iloc[0, 1]
print(f"📊 Correlación SERVQUAL-Satisfacción: {corr_servqual_sat:.3f}")

print("\n✅ 3 scatter plots con regresión completados")

---
# ✅ SECCIÓN 6: Heatmaps y Correlaciones

**Objetivo:** Crear 4 heatmaps para visualizar matrices de correlación y tablas pivote.

Los heatmaps son ideales para identificar patrones en datos multivariados, especialmente correlaciones y relaciones entre categorías.

## Contenido:
1. **Heatmap de Correlaciones Completo**: Todas las variables numéricas
2. **Heatmap Anotado**: Solo variables clave con valores impresos
3. **Heatmap Pivot**: Satisfacción promedio por Giro × Puesto
4. **Heatmap de Frecuencias**: Giro × Estado (Top 10)

**Total:** 4 heatmaps interactivos

In [ ]:
## 6.1 HEATMAP DE CORRELACIONES COMPLETO

print("="*80)
print("SECCIÓN 6.1: HEATMAP DE CORRELACIONES - TODAS LAS VARIABLES NUMÉRICAS")
print("="*80)

# Gráfico 1: Matriz de correlación de todas las variables numéricas
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Excluir variables derivadas redundantes para mejor visualización
vars_correlacion = ['AT_1', 'AT_2', 'FI_1', 'FI_2', 'FI_3', 'R_1', 'R_2', 'R_3',
                     'E_1', 'E_2', 'E_3', 'E_4', 'C_1', 'D_1', 'R_12', 'Info', 'Años',
                     'Tangibles', 'Fiabilidad', 'Respuesta', 'Empatia', 'SERVQUAL_Total',
                     'Indice_Excelencia']

corr_matrix = df[vars_correlacion].corr()

fig1 = px.imshow(
    corr_matrix,
    title='🔥 Matriz de Correlación Completa (23 Variables)',
    labels=dict(x="Variable", y="Variable", color="Correlación"),
    color_continuous_scale='RdBu_r',  # Rojo-Blanco-Azul invertido
    color_continuous_midpoint=0,  # Centrar en 0
    aspect='auto',
    height=800,
    width=850
)
fig1.update_layout(
    font=dict(size=9),
    xaxis={'side': 'bottom'}
)
fig1.update_xaxes(tickangle=-45)
fig1.show()

print("✅ Heatmap de correlación completo creado")
print(f"   Variables analizadas: {len(vars_correlacion)}")

---
# ✅ SECCIÓN 7: Gráficos de Distribución Avanzados

**Objetivo:** Crear 5 visualizaciones de distribuciones bidimensionales.

Estas técnicas avanzadas muestran la densidad de puntos en el espacio 2D, útiles para identificar concentraciones, clusters, y patrones no lineales.

## Contenido:
1. **Density Heatmap 2D** (2 gráficos): Mapas de calor de densidad
2. **Density Contour** (1 gráfico): Contornos de densidad con marginales
3. **Histograma 2D** (1 gráfico): Binning bidimensional
4. **Scatter con Rug Plot** (1 gráfico): Distribuciones marginales

**Total:** 5 gráficos de distribución avanzada

In [ ]:
## 7.1 DENSITY HEATMAP 2D (2 gráficos)

print("="*80)
print("SECCIÓN 7.1: DENSITY HEATMAP 2D - MAPAS DE CALOR DE DENSIDAD")
print("="*80)

# Gráfico 1: Densidad 2D - Empatía vs Satisfacción
fig1 = px.density_heatmap(
    df,
    x='Empatia',
    y='D_1',
    title='🔥 Densidad 2D: Empatía vs Satisfacción',
    labels={'Empatia': 'Empatía (1-5)', 'D_1': 'Satisfacción (1-10)'},
    color_continuous_scale='Viridis',
    nbinsx=20,
    nbinsy=20,
    height=500
)
fig1.update_layout(
    coloraxis_colorbar=dict(title="Densidad")
)
fig1.show()

# Gráfico 2: Densidad 2D - Años vs NPS (con marginales)
fig2 = px.density_heatmap(
    df,
    x='Años',
    y='R_12',
    title='📊 Densidad 2D: Antigüedad vs NPS (con distribuciones marginales)',
    labels={'Años': 'Años como benefactor', 'R_12': 'NPS (1-10)'},
    color_continuous_scale='Plasma',
    nbinsx=25,
    nbinsy=10,
    marginal_x='histogram',
    marginal_y='histogram',
    height=550
)
fig2.update_layout(
    coloraxis_colorbar=dict(title="Densidad")
)
fig2.show()

print("\n✅ 2 density heatmaps 2D completados")

In [ ]:
## 7.4 SCATTER CON RUG PLOT (1 gráfico)

print("\n" + "="*80)
print("SECCIÓN 7.4: SCATTER CON RUG PLOT - DISTRIBUCIONES MARGINALES")
print("="*80)

# Gráfico 5: Scatter con rug - Años vs NPS
fig5 = px.scatter(
    df,
    x='Años',
    y='R_12',
    title='🔵 Scatter Plot con Rug: Antigüedad vs NPS',
    labels={'Años': 'Años como benefactor', 'R_12': 'NPS (1-10)'},
    color='NPS_Categoria',
    color_discrete_map={'Detractor': '#d62728', 'Pasivo': '#ff7f0e', 'Promotor': '#2ca02c'},
    marginal_x='rug',
    marginal_y='rug',
    hover_data=['Giro', 'Estado'],
    opacity=0.6,
    height=550
)
fig5.update_traces(marker=dict(size=8))
fig5.show()

print("\n✅ 1 scatter con rug plot completado")

print("\n" + "="*80)
print("🎉 SECCIÓN 7 COMPLETADA: 5 GRÁFICOS DE DISTRIBUCIÓN AVANZADA CREADOS")
print("="*80)
print("\n📊 RESUMEN:")
print("   • 2 Density Heatmaps 2D (Empatía-Satisfacción, Años-NPS)")
print("   • 1 Density Contour con marginales (SERVQUAL-Satisfacción)")
print("   • 1 Histograma 2D (Empatía-Satisfacción)")
print("   • 1 Scatter con Rug Plot (Años-NPS)")
print("\n" + "="*80)
print("🎉 ITERACIÓN 9 COMPLETADA: SECCIONES 6-7 (9 VISUALIZACIONES)")
print("="*80)
print("\n📊 RESUMEN DE ITERACIÓN 9:")
print("   SECCIÓN 6: 4 Heatmaps")
print("   SECCIÓN 7: 5 Distribuciones Avanzadas")
print("\n🚀 LISTO PARA ITERACIÓN 10: Series + Burbujas + Tendencias")
print("="*80)

In [ ]:
## 7.3 HISTOGRAMA 2D (1 gráfico)

print("\n" + "="*80)
print("SECCIÓN 7.3: HISTOGRAMA 2D - BINNING BIDIMENSIONAL")
print("="*80)

# Gráfico 4: Histograma 2D - Empatía vs Satisfacción
fig4 = go.Figure(go.Histogram2d(
    x=df['Empatia'],
    y=df['D_1'],
    colorscale='Blues',
    nbinsx=15,
    nbinsy=15,
    colorbar=dict(title="Frecuencia")
))

fig4.update_layout(
    title='📊 Histograma 2D: Empatía vs Satisfacción (bins 15×15)',
    xaxis_title='Empatía (1-5)',
    yaxis_title='Satisfacción (1-10)',
    height=500,
    width=650
)
fig4.show()

print("\n✅ 1 histograma 2D completado")

In [ ]:
## 7.2 DENSITY CONTOUR (1 gráfico)

print("\n" + "="*80)
print("SECCIÓN 7.2: DENSITY CONTOUR - CONTORNOS DE DENSIDAD")
print("="*80)

# Gráfico 3: Contorno de densidad - SERVQUAL Total vs Satisfacción (con marginales)
fig3 = px.density_contour(
    df,
    x='SERVQUAL_Total',
    y='D_1',
    title='🌀 Contornos de Densidad: SERVQUAL Total vs Satisfacción (con marginales)',
    labels={'SERVQUAL_Total': 'SERVQUAL Total (1-5)', 'D_1': 'Satisfacción (1-10)'},
    color='Giro',
    marginal_x='violin',
    marginal_y='box',
    height=550
)
fig3.update_traces(contours_coloring='fill', contours_showlabels=True)
fig3.show()

print("\n✅ 1 density contour completado")

In [ ]:
## 6.2 HEATMAP ANOTADO - VARIABLES CLAVE

print("\n" + "="*80)
print("SECCIÓN 6.2: HEATMAP ANOTADO - VARIABLES CLAVE CON VALORES")
print("="*80)

# Gráfico 2: Heatmap solo de variables clave con anotaciones
vars_clave = ['D_1', 'R_12', 'C_1', 'Info', 'Tangibles', 'Fiabilidad', 
              'Respuesta', 'Empatia', 'SERVQUAL_Total', 'Años', 'Indice_Excelencia']

corr_clave = df[vars_clave].corr()

# Crear anotaciones con valores redondeados
annotations = []
for i, row in enumerate(corr_clave.index):
    for j, col in enumerate(corr_clave.columns):
        annotations.append(
            dict(
                x=j,
                y=i,
                text=str(round(corr_clave.iloc[i, j], 2)),
                showarrow=False,
                font=dict(color='white' if abs(corr_clave.iloc[i, j]) > 0.5 else 'black', size=10)
            )
        )

fig2 = px.imshow(
    corr_clave,
    title='🔢 Matriz de Correlación: Variables Clave (con valores anotados)',
    labels=dict(x="Variable", y="Variable", color="Correlación"),
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    aspect='auto',
    height=600,
    width=650
)
fig2.update_layout(
    annotations=annotations,
    font=dict(size=10)
)
fig2.update_xaxes(tickangle=-45)
fig2.show()

# Mostrar top 5 correlaciones (excluyendo diagonal)
print("\n📊 Top 5 Correlaciones más fuertes:")
corr_pairs = corr_clave.where(np.triu(np.ones(corr_clave.shape), k=1).astype(bool))
corr_pairs = corr_pairs.stack().reset_index()
corr_pairs.columns = ['Variable_1', 'Variable_2', 'Correlacion']
corr_pairs['Correlacion_Abs'] = corr_pairs['Correlacion'].abs()
top_5 = corr_pairs.nlargest(5, 'Correlacion_Abs')[['Variable_1', 'Variable_2', 'Correlacion']]
print(top_5.to_string(index=False))

print("\n✅ Heatmap anotado creado")

In [ ]:
## 5.5 GRÁFICOS DE BARRAS AGRUPADAS (2 gráficos)

print("\n" + "="*80)
print("SECCIÓN 5.5: BARRAS AGRUPADAS - COMPARACIONES CATEGÓRICAS")
print("="*80)

# Gráfico 11: Satisfacción promedio por Giro
satisfaccion_por_giro = df.groupby('Giro')['D_1'].mean().reset_index()
satisfaccion_por_giro.columns = ['Giro', 'Satisfaccion_Promedio']
satisfaccion_por_giro = satisfaccion_por_giro.sort_values('Satisfaccion_Promedio', ascending=False)

fig11 = px.bar(
    satisfaccion_por_giro,
    x='Giro',
    y='Satisfaccion_Promedio',
    title='📊 Satisfacción Promedio por Giro Empresarial',
    labels={'Giro': 'Giro', 'Satisfaccion_Promedio': 'Satisfacción Promedio (1-10)'},
    color='Satisfaccion_Promedio',
    color_continuous_scale='RdYlGn',
    text='Satisfaccion_Promedio'
)
fig11.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig11.update_layout(
    height=450,
    xaxis_tickangle=-45,
    yaxis_range=[0, 10]
)
fig11.show()

# Gráfico 12: Dimensiones SERVQUAL por Giro (barmode='group')
# Preparar datos
dimensiones_por_giro = df.groupby('Giro')[['Tangibles', 'Fiabilidad', 'Respuesta', 'Empatia']].mean().reset_index()
dimensiones_melted = dimensiones_por_giro.melt(
    id_vars='Giro',
    value_vars=['Tangibles', 'Fiabilidad', 'Respuesta', 'Empatia'],
    var_name='Dimension',
    value_name='Puntaje_Promedio'
)

fig12 = px.bar(
    dimensiones_melted,
    x='Giro',
    y='Puntaje_Promedio',
    color='Dimension',
    title='🔄 Comparación de Dimensiones SERVQUAL por Giro (barras agrupadas)',
    labels={'Giro': 'Giro Empresarial', 'Puntaje_Promedio': 'Puntaje Promedio (1-5)', 'Dimension': 'Dimensión SERVQUAL'},
    barmode='group',  # Barras agrupadas
    text='Puntaje_Promedio'
)
fig12.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig12.update_layout(
    height=500,
    xaxis_tickangle=-45,
    yaxis_range=[0, 5.5]
)
fig12.show()

print("\n✅ 2 gráficos de barras agrupadas completados")

In [ ]:
## 5.4 VIOLIN PLOTS POR GRUPOS (2 gráficos)

print("\n" + "="*80)
print("SECCIÓN 5.4: VIOLIN PLOTS POR GRUPOS - DENSIDADES COMPARATIVAS")
print("="*80)

# Gráfico 9: Satisfacción por Giro (con box y puntos)
fig9 = px.violin(
    df,
    x='Giro',
    y='D_1',
    title='🎻 Distribución de Satisfacción por Giro (con densidad)',
    labels={'Giro': 'Giro Empresarial', 'D_1': 'Satisfacción (1-10)'},
    color='Giro',
    box=True,  # Incluir box plot interno
    points='outliers'  # Mostrar solo outliers
)
fig9.update_layout(
    height=500,
    xaxis_tickangle=-45,
    showlegend=False
)
fig9.show()

# Gráfico 10: NPS por Categoría de Satisfacción
fig10 = px.violin(
    df,
    x='Categoria_Satisfaccion',
    y='R_12',
    title='🎯 Distribución de NPS por Categoría de Satisfacción',
    labels={'Categoria_Satisfaccion': 'Categoría de Satisfacción', 'R_12': 'NPS (1-10)'},
    color='Categoria_Satisfaccion',
    color_discrete_map={'Baja': '#d62728', 'Media': '#ff7f0e', 'Alta': '#2ca02c'},
    box=True,
    points='all',
    category_orders={'Categoria_Satisfaccion': ['Baja', 'Media', 'Alta']}
)
fig10.update_layout(
    height=450,
    showlegend=False
)
fig10.show()

print("\n📊 NPS promedio por Categoría de Satisfacción:")
print(df.groupby('Categoria_Satisfaccion')['R_12'].agg(['mean', 'median', 'count']).round(2))

print("\n✅ 2 violin plots por grupos completados")

In [ ]:
## 5.3 BOX PLOTS POR GRUPOS (4 gráficos)

print("\n" + "="*80)
print("SECCIÓN 5.3: BOX PLOTS POR GRUPOS - COMPARACIÓN DE DISTRIBUCIONES")
print("="*80)

# Gráfico 5: Satisfacción por Giro
fig5 = px.box(
    df,
    x='Giro',
    y='D_1',
    title='🏢 Satisfacción General por Giro Empresarial',
    labels={'Giro': 'Giro', 'D_1': 'Satisfacción (1-10)'},
    color='Giro',
    points='outliers',
    notched=False
)
fig5.update_layout(
    height=450,
    xaxis_tickangle=-45,
    showlegend=False
)
fig5.show()

# Estadísticas por grupo
print("\n📊 Satisfacción promedio por Giro:")
print(df.groupby('Giro')['D_1'].agg(['mean', 'median', 'std']).round(2))

# Gráfico 6: NPS por Puesto (con notches para IC de mediana)
fig6 = px.box(
    df,
    x='Puesto',
    y='R_12',
    title='👔 NPS por Puesto del Encuestado (con intervalos de confianza)',
    labels={'Puesto': 'Puesto', 'R_12': 'NPS (1-10)'},
    color='Puesto',
    points='all',  # Mostrar todos los puntos
    notched=True  # Agregar notches para IC de mediana
)
fig6.update_layout(
    height=500,
    xaxis_tickangle=-45,
    showlegend=False
)
fig6.show()

# Gráfico 7: Calidad Percibida por Región
fig7 = px.box(
    df,
    x='Region',
    y='C_1',
    title='🗺️ Calidad Percibida por Región Geográfica',
    labels={'Region': 'Región de México', 'C_1': 'Calidad Percibida (1-5)'},
    color='Region',
    points='outliers'
)
fig7.update_layout(
    height=450,
    showlegend=False
)
fig7.show()

print("\n📊 Calidad promedio por Región:")
print(df.groupby('Region')['C_1'].agg(['mean', 'count']).round(2).sort_values('mean', ascending=False))

# Gráfico 8: Índice de Excelencia por Grupo de Antigüedad
fig8 = px.box(
    df,
    x='Grupo_Antiguedad',
    y='Indice_Excelencia',
    title='⏳ Índice de Excelencia por Antigüedad como Benefactor',
    labels={'Grupo_Antiguedad': 'Grupo de Antigüedad', 'Indice_Excelencia': 'Índice de Excelencia (0-100)'},
    color='Grupo_Antiguedad',
    points='outliers',
    category_orders={'Grupo_Antiguedad': ['Nuevo', 'Intermedio', 'Fiel', 'Muy Fiel']}
)
fig8.update_layout(
    height=450,
    showlegend=False
)
fig8.show()

print("\n✅ 4 box plots por grupos completados")

In [ ]:
## 5.2 SCATTER MATRIX (1 gráfico)

print("\n" + "="*80)
print("SECCIÓN 5.2: SCATTER MATRIX - MATRIZ DE DISPERSIÓN MULTIVARIADA")
print("="*80)

# Gráfico 4: Matriz de dispersión de variables clave
variables_clave = ['D_1', 'R_12', 'C_1', 'Info', 'SERVQUAL_Total', 'Años']

fig4 = px.scatter_matrix(
    df,
    dimensions=variables_clave,
    title='🔢 Matriz de Dispersión: Variables Clave del Estudio',
    color='Giro',
    opacity=0.6,
    hover_data=['Estado'],
    height=900,
    width=900
)
fig4.update_traces(diagonal_visible=True, showupperhalf=False, marker=dict(size=4))
fig4.update_layout(
    font=dict(size=10)
)
fig4.show()

print("\n✅ Scatter matrix completado")
print("💡 Esta matriz muestra todas las relaciones bivariadas entre 6 variables clave")

In [ ]:
## 4.5 PIE CHARTS Y DONUT CHARTS (2 gráficos)

print("\n" + "="*80)
print("SECCIÓN 4.5: PIE/DONUT CHARTS - COMPOSICIÓN PORCENTUAL")
print("="*80)

# Gráfico 12: Composición por Giro (donut chart)
giro_counts = df['Giro'].value_counts().reset_index()
giro_counts.columns = ['Giro', 'Cantidad']

fig12 = px.pie(
    giro_counts,
    names='Giro',
    values='Cantidad',
    title='🍩 Composición de Empresas por Giro (Donut Chart)',
    hole=0.4,  # Crear efecto donut
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig12.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Empresas: %{value}<br>Porcentaje: %{percent}<extra></extra>'
)
fig12.update_layout(height=500)
fig12.show()

# Gráfico 13: Niveles de Calidad Percibida
calidad_counts = df['Nivel_Calidad'].value_counts().reset_index()
calidad_counts.columns = ['Nivel_Calidad', 'Cantidad']

# Colores por nivel
colores_calidad = {
    'Deficiente': '#d62728',
    'Aceptable': '#ff7f0e', 
    'Bueno': '#2ca02c',
    'Excelente': '#1f77b4'
}

fig13 = px.pie(
    calidad_counts,
    names='Nivel_Calidad',
    values='Cantidad',
    title='⭐ Distribución de Niveles de Calidad Percibida (C_1)',
    color='Nivel_Calidad',
    color_discrete_map=colores_calidad
)
fig13.update_traces(
    textposition='auto',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Empresas: %{value}<br>Porcentaje: %{percent}<extra></extra>'
)
fig13.update_layout(height=500)
fig13.show()

print("\n✅ 2 pie/donut charts completados")

In [ ]:
## 4.4 GRÁFICOS DE BARRAS (3 gráficos)

print("\n" + "="*80)
print("SECCIÓN 4.4: GRÁFICOS DE BARRAS - FRECUENCIAS CATEGÓRICAS")
print("="*80)

# Gráfico 9: Distribución por Giro
giro_counts = df['Giro'].value_counts().reset_index()
giro_counts.columns = ['Giro', 'Frecuencia']

fig9 = px.bar(
    giro_counts,
    x='Giro',
    y='Frecuencia',
    title='🏢 Distribución de Empresas por Giro',
    labels={'Giro': 'Sector Empresarial', 'Frecuencia': 'Número de Empresas'},
    color='Frecuencia',
    color_continuous_scale='Blues',
    text='Frecuencia'
)
fig9.update_traces(textposition='outside')
fig9.update_layout(
    height=450,
    xaxis_tickangle=-45,
    showlegend=False
)
fig9.show()

# Gráfico 10: Distribución por NPS Categoría
nps_counts = df['NPS_Categoria'].value_counts().reset_index()
nps_counts.columns = ['NPS_Categoria', 'Frecuencia']

# Ordenar correctamente
orden_nps = ['Detractor', 'Pasivo', 'Promotor']
nps_counts['NPS_Categoria'] = pd.Categorical(nps_counts['NPS_Categoria'], categories=orden_nps, ordered=True)
nps_counts = nps_counts.sort_values('NPS_Categoria')

# Colores por categoría
colores_nps = {'Detractor': '#d62728', 'Pasivo': '#ff7f0e', 'Promotor': '#2ca02c'}

fig10 = px.bar(
    nps_counts,
    x='NPS_Categoria',
    y='Frecuencia',
    title='🎯 Distribución de Categorías NPS (Net Promoter Score)',
    labels={'NPS_Categoria': 'Categoría NPS', 'Frecuencia': 'Número de Benefactores'},
    color='NPS_Categoria',
    color_discrete_map=colores_nps,
    text='Frecuencia'
)
fig10.update_traces(textposition='outside')
fig10.update_layout(
    height=450,
    showlegend=False
)
fig10.show()

# Gráfico 11: Top 10 Estados
top_estados = df['Estado'].value_counts().head(10).reset_index()
top_estados.columns = ['Estado', 'Frecuencia']

fig11 = px.bar(
    top_estados,
    y='Estado',
    x='Frecuencia',
    title='📍 Top 10 Estados con Más Empresas Benefactoras',
    labels={'Estado': 'Estado', 'Frecuencia': 'Número de Empresas'},
    orientation='h',
    color='Frecuencia',
    color_continuous_scale='Greens',
    text='Frecuencia'
)
fig11.update_traces(textposition='outside')
fig11.update_layout(
    height=450,
    yaxis={'categoryorder': 'total ascending'}
)
fig11.show()

print("\n✅ 3 gráficos de barras completados")

In [ ]:
## 4.3 VIOLIN PLOTS (2 gráficos)

print("\n" + "="*80)
print("SECCIÓN 4.3: VIOLIN PLOTS - DISTRIBUCIÓN + DENSIDAD")
print("="*80)

# Gráfico 7: Violin plot de Satisfacción con box interno y puntos
fig7 = px.violin(
    df,
    y='D_1',
    title='🎻 Violin Plot de Satisfacción General (D_1)',
    labels={'D_1': 'Satisfacción (1-10)'},
    box=True,  # Incluir box plot interno
    points='all',  # Mostrar todos los puntos
    color_discrete_sequence=['#e377c2']
)
fig7.update_layout(
    height=450,
    showlegend=False
)
fig7.show()

# Gráfico 8: Violin plot comparativo (Satisfacción vs NPS)
comparativo_df = pd.DataFrame({
    'Satisfacción (D_1)': df['D_1'],
    'NPS (R_12)': df['R_12']
}).melt(var_name='Variable', value_name='Puntaje')

fig8 = px.violin(
    comparativo_df,
    x='Variable',
    y='Puntaje',
    title='🎻 Comparación: Distribución de Satisfacción vs NPS',
    labels={'Puntaje': 'Puntaje (1-10)', 'Variable': ''},
    color='Variable',
    box=True,
    points='outliers'
)
fig8.update_layout(
    height=450,
    showlegend=False
)
fig8.show()

print("\n✅ 2 violin plots completados")

In [ ]:
## 4.2 BOX PLOTS (2 gráficos)

print("\n" + "="*80)
print("SECCIÓN 4.2: BOX PLOTS - IDENTIFICACIÓN DE OUTLIERS Y ESTADÍSTICOS")
print("="*80)

# Gráfico 5: Box plot de las 4 dimensiones SERVQUAL
dimensiones_df = df[['Tangibles', 'Fiabilidad', 'Respuesta', 'Empatia']].melt(
    var_name='Dimensión',
    value_name='Puntaje'
)

fig5 = px.box(
    dimensiones_df,
    x='Dimensión',
    y='Puntaje',
    title='📦 Distribución de las 4 Dimensiones SERVQUAL (con outliers)',
    labels={'Puntaje': 'Puntaje (1-5)', 'Dimensión': 'Dimensión SERVQUAL'},
    color='Dimensión',
    points='outliers',  # Mostrar solo outliers
    notched=True  # Agregar notch para IC de la mediana
)
fig5.update_layout(
    height=450,
    showlegend=False
)
fig5.show()

# Gráfico 6: Box plot horizontal de Satisfacción con todos los puntos
fig6 = px.box(
    df,
    y='D_1',
    title='📊 Box Plot Horizontal de Satisfacción (D_1) - Con Media y Puntos',
    labels={'D_1': 'Satisfacción (1-10)'},
    points='all',  # Mostrar todos los puntos
    color_discrete_sequence=['#17becf']
)
fig6.update_layout(
    height=400,
    showlegend=False
)
# Agregar línea de media
fig6.add_hline(y=df['D_1'].mean(), line_dash="dash", line_color="red",
               annotation_text=f"Media: {df['D_1'].mean():.2f}",
               annotation_position="right")
fig6.show()

print("\n✅ 2 box plots completados")

---
# 🟡 SECCIÓN 5: Visualizaciones Bivariadas

## TODO: COMPLETAR EN ITERACIÓN 8

### 5.1 Scatter Plots con Regresión (3 gráficos)
- Empatía vs Satisfacción (con trendline OLS)
- Años vs NPS (con trendline)
- SERVQUAL Total vs Satisfacción (trendline LOWESS)

### 5.2 Scatter Matrix (1 gráfico)
- Matriz de dispersión de variables clave (D_1, R_12, C_1, Info, SERVQUAL, Años)

### 5.3 Box Plots por Grupos (4 gráficos)
- Satisfacción por Giro
- NPS por Puesto (con notches)
- Calidad por Región
- Índice de Excelencia por Antigüedad

### 5.4 Violin Plots por Grupos (2 gráficos)
- Satisfacción por Giro (con box y puntos)
- NPS por Categoría de Satisfacción

### 5.5 Gráficos de Barras Agrupadas (2 gráficos)
- Satisfacción promedio por Giro
- Dimensiones SERVQUAL por Giro (barmode='group')

### 5.6 Barras Apiladas (1 gráfico)
- Composición de NPS por Giro (100% stacked)

### 5.7 Strip Plots (1 gráfico)
- Distribución individual de Satisfacción por Giro

**Total:** ~14 visualizaciones bivariadas  
**Código estimado:** ~140 líneas

In [ ]:
# TODO: Completar Sección 5 en Iteración 8
print("🟡 Sección 5: Pendiente de completar")
print("   Se crearán ~14 visualizaciones bivariadas")

---
# 🟡 SECCIÓN 6: Heatmaps y Correlaciones

## TODO: COMPLETAR EN ITERACIÓN 9

### 6.1 Heatmap de Correlaciones (1 gráfico)
- Matriz de correlación de todas las variables numéricas
- Colorscale RdBu centrado en 0

### 6.2 Heatmap Anotado (1 gráfico)
- Solo variables clave (D_1, R_12, C_1, Info, dimensiones SERVQUAL)

### 6.3 Heatmap Promedio Satisfacción (1 gráfico)
- Tabla pivot: Giro × Puesto (valores = D_1 promedio)

### 6.4 Heatmap de Frecuencias (1 gráfico)
- Giro × Estado (Top 10 estados)

**Total:** 4 heatmaps  
**Código estimado:** ~60 líneas

In [ ]:
# TODO: Completar Sección 6 en Iteración 9
print("🟡 Sección 6: Pendiente de completar")

---
# 🟡 SECCIÓN 7: Gráficos de Distribución Avanzados

## TODO: COMPLETAR EN ITERACIÓN 9

### 7.1 Distribución 2D - Density Heatmap (2 gráficos)
- Densidad 2D: Empatía vs Satisfacción
- Densidad 2D: Años vs NPS (con marginales)

### 7.2 Density Contour (1 gráfico)
- Contorno de densidad: SERVQUAL Total vs Satisfacción (con marginales)

### 7.3 Histogramas 2D (1 gráfico)
- Histograma 2D: Empatía vs Satisfacción

### 7.4 Distribuciones con Rug Plot (1 gráfico)
- Scatter con rug: Años vs NPS

**Total:** 5 gráficos de distribución avanzada  
**Código estimado:** ~50 líneas

In [ ]:
# TODO: Completar Sección 7 en Iteración 9
print("🟡 Sección 7: Pendiente de completar")

---
# 🟡 SECCIÓN 8: Gráficos de Series y Comparaciones

## TODO: COMPLETAR EN ITERACIÓN 10

### 8.1 Parallel Coordinates (2 gráficos)
- Perfiles multivariados por Giro
- Perfiles por nivel de Satisfacción

### 8.2 Parallel Categories (1 gráfico)
- Flujo: Giro → Puesto → NPS Categoría

### 8.3 Radar Chart / Spider Plot (2 gráficos)
- Perfil SERVQUAL promedio general
- Comparación Top 3 Giros

**Total:** 5 gráficos  
**Código estimado:** ~60 líneas

In [ ]:
# TODO: Completar Sección 8 en Iteración 10
print("🟡 Sección 8: Pendiente de completar")

---
# 🟡 SECCIÓN 9: Gráficos de Burbuja y 3D

## TODO: COMPLETAR EN ITERACIÓN 10

### 9.1 Bubble Chart (2 gráficos)
- X=Empatía, Y=Satisfacción, Size=Años, Color=Giro
- X=SERVQUAL, Y=Satisfacción, Size=NPS, Color=Índice

### 9.2 Scatter 3D (2 gráficos)
- Empatía, Satisfacción, NPS (color=Giro, size=Años)
- Tangibles, Fiabilidad, Empatía (color=Satisfacción)

**Total:** 4 gráficos  
**Código estimado:** ~40 líneas

In [ ]:
# TODO: Completar Sección 9 en Iteración 10
print("🟡 Sección 9: Pendiente de completar")

---
# 🟡 SECCIÓN 10: Gráficos de Tendencia y Series

## TODO: COMPLETAR EN ITERACIÓN 10

### 10.1 Line Charts por Grupo (2 gráficos)
- Tendencia de satisfacción promedio por antigüedad
- Tendencias por Giro

### 10.2 Area Chart (1 gráfico)
- Crecimiento acumulado de benefactores

**Total:** 3 gráficos  
**Código estimado:** ~30 líneas

In [ ]:
# TODO: Completar Sección 10 en Iteración 10
print("🟡 Sección 10: Pendiente de completar")

---
# 🟡 SECCIÓN 11: Gráficos Estadísticos Avanzados

## TODO: COMPLETAR EN ITERACIÓN 11

### 11.1 Distplot (2 gráficos)
- Distribución de Satisfacción con curva de densidad + rug
- Comparación Educación vs Empresa

### 11.2 QQ Plot (1 gráfico)
- Verificar normalidad de Satisfacción (D_1)

### 11.3 ECDF (2 gráficos)
- Función de distribución acumulada: Satisfacción
- ECDF por Giro

**Total:** 5 gráficos  
**Código estimado:** ~50 líneas

In [ ]:
# TODO: Completar Sección 11 en Iteración 11
print("🟡 Sección 11: Pendiente de completar")

---
# 🟡 SECCIÓN 12: Dashboards Interactivos con Subplots

## TODO: COMPLETAR EN ITERACIÓN 11

### 12.1 Dashboard de KPIs (1 dashboard con 4 subplots)
- Subplot 2×2:
  - Satisfacción por Giro (barras)
  - NPS por Puesto (box plot)
  - Distribución de Años (histograma)
  - Índice de Excelencia (violin)

### 12.2 Dashboard Comparativo (1 dashboard con 4 subplots)
- Distribución de 4 dimensiones SERVQUAL (histogramas)

**Total:** 2 dashboards integrados  
**Código estimado:** ~60 líneas

In [ ]:
# TODO: Completar Sección 12 en Iteración 11
print("🟡 Sección 12: Pendiente de completar")

---
# 🟡 SECCIÓN 13: BONUS - Profilers Automáticos

## TODO: COMPLETAR EN ITERACIÓN 12

Esta sección mostrará herramientas de profiling automático:

### 13.1 Pandas Profiling (ydata-profiling)
```python
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title='Reporte Teletón', explorative=True)
profile.to_file("teleton_profiling_report.html")
profile.to_widgets()  # Mostrar en notebook
```

### 13.2 Sweetviz
```python
import sweetviz as sv

report = sv.analyze(df)
report.show_html('teleton_sweetviz_report.html')
```

### 13.3 AutoViz
```python
from autoviz.AutoViz_Class import AutoViz_Class

AV = AutoViz_Class()
dft = AV.AutoViz(filename='', dfte=df, depVar='D_1')
```

### 13.4 D-Tale
```python
import dtale

d = dtale.show(df)
d.open_browser()
```

**Archivos generados:**
- `teleton_profiling_report.html`
- `teleton_sweetviz_report.html`

**Código estimado:** ~40 líneas

In [ ]:
# TODO: Completar Sección 13 en Iteración 12
print("🟡 Sección 13: Pendiente de completar")
print("   Se instalarán y ejecutarán 4 profilers automáticos")

---
# 🟡 SECCIÓN 14: Exportación para BI Tools

## TODO: COMPLETAR EN ITERACIÓN 12

### 14.1 Preparar para Google Data Studio / Looker Studio
- Seleccionar columnas relevantes
- Exportar a CSV optimizado
- Instrucciones de carga

### 14.2 Preparar para BigQuery
- Convertir categóricas a string
- Generar schema SQL
- Exportar CSV

### 14.3 Preparar para Tableau
- Exportar CSV
- Exportar Excel

**Archivos a generar:**
- `datos_procesados/teleton_para_looker.csv`
- `datos_procesados/teleton_para_bigquery.csv`
- `datos_procesados/teleton_para_tableau.csv`
- `datos_procesados/teleton_para_tableau.xlsx`

**Código estimado:** ~80 líneas

In [ ]:
# TODO: Completar Sección 14 en Iteración 12
print("🟡 Sección 14: Pendiente de completar")
print("   Se generarán 4 archivos de exportación para BI tools")

---
# 🟡 SECCIÓN 15: Conclusiones y Próximos Pasos

## TODO: COMPLETAR EN ITERACIÓN 12

### Resumen del Notebook 2
- Total de visualizaciones creadas: ~50+
- Datos enriquecidos: [número] columnas → [número] columnas
- Archivos exportados: 4 para BI tools

### Próximos Pasos
1. Cargar datos en Looker Studio
2. Crear dashboard interactivo
3. Presentar insights a stakeholders

**Formato:** Markdown con resumen ejecutivo

## 🎯 Resumen del Notebook 2

### Datos Enriquecidos
- **Columnas originales:** 20
- **Columnas enriquecidas:** [TODO]
- **Variables categóricas derivadas:** [TODO]
- **Índices calculados:** [TODO]

### Visualizaciones Creadas
Total de ~50+ gráficos con Plotly Express:
- [TODO] Completar lista al terminar iteraciones

### Profilers Automáticos
- [TODO] ydata-profiling
- [TODO] Sweetviz
- [TODO] AutoViz
- [TODO] D-Tale

### Exportaciones Generadas
- [TODO] teleton_para_looker.csv
- [TODO] teleton_para_bigquery.csv
- [TODO] teleton_para_tableau.csv/.xlsx

### Próximos Pasos
1. Cargar en Looker Studio
2. Crear dashboard interactivo
3. Presentar a stakeholders de Teletón

---

**¡Dashboard listo para producción! 🎉**

---
# 📝 Resumen del Estado del Notebook

## ✅ Secciones Completas (2/15)
1. ✅ Setup y Carga de Datos
2. ✅ Diccionario de Datos

## 🟡 Secciones Pendientes (13/15)
3. 🟡 Enriquecimiento de Datos → **Iteración 6**
4. 🟡 Visualizaciones Univariadas → **Iteración 7**
5. 🟡 Visualizaciones Bivariadas → **Iteración 8**
6. 🟡 Heatmaps y Correlaciones → **Iteración 9**
7. 🟡 Distribuciones Avanzadas → **Iteración 9**
8. 🟡 Series y Comparaciones → **Iteración 10**
9. 🟡 Burbuja y 3D → **Iteración 10**
10. 🟡 Tendencias → **Iteración 10**
11. 🟡 Estadísticos Avanzados → **Iteración 11**
12. 🟡 Dashboards Subplots → **Iteración 11**
13. 🟡 Profilers Automáticos → **Iteración 12**
14. 🟡 Exportación BI Tools → **Iteración 12**
15. 🟡 Conclusiones → **Iteración 12**

---

## 🚀 Próximos Pasos

Para completar este notebook, ejecuta las iteraciones en orden:

```
Iteración 6: Enriquecimiento de Datos (~80 líneas)
Iteración 7: Visualizaciones Univariadas (~150 líneas)
Iteración 8: Visualizaciones Bivariadas (~140 líneas)
Iteración 9: Heatmaps + Distribuciones Avanzadas (~110 líneas)
Iteración 10: Series + Burbuja/3D + Tendencias (~130 líneas)
Iteración 11: Estadísticos + Dashboards (~110 líneas)
Iteración 12: Profilers + Exportación + Conclusiones (~120 líneas)
```

**Total estimado:** ~840 líneas adicionales de código  
**Total visualizaciones:** ~50+ gráficos interactivos con Plotly

---

**Notebook creado:** Fase 1 - Estructura Completa  
**Estado actual:** Ejecutable pero incompleto (secciones 1-2 funcionales)  
**Dataset:** 274 empresas benefactoras × 20 variables (+ derivadas)  
**Listo para:** Iteración 6